<a href="https://colab.research.google.com/github/rpark3/ECON3916-Statistical-Machine-Learning/blob/main/Lab%2012/Lab_12_Architecting_the_Prediction_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.tools.eval_measures import rmse
import matplotlib.pyplot as plt

# Step 1: Ingestion from external source
url = 'https://raw.githubusercontent.com/rpark3/ECON3916-Statistical-Machine-Learning/refs/heads/main/Data/Zillow_ZHVI_2026_Micro.csv'
df = pd.read_csv(url)

In [11]:
# Step 2: Defining the formula
# Utilizing the R-style patsy formula interface allows for elegant, readable model specification
formula = 'Home_Value ~ Square_Footage + Property_Age + Distance_to_Transit + School_District_Rating'

In [12]:
# Step 3: Fitting the model and printing the summary
model = smf.ols(formula = formula, data = df)
results = model.fit()
print(results.summary())

                            OLS Regression Results                            
Dep. Variable:             Home_Value   R-squared:                       0.766
Model:                            OLS   Adj. R-squared:                  0.765
Method:                 Least Squares   F-statistic:                     542.5
Date:                Mon, 16 Mar 2026   Prob (F-statistic):          2.81e-309
Time:                        19:54:36   Log-Likelihood:                -12072.
No. Observations:                1000   AIC:                         2.416e+04
Df Residuals:                     993   BIC:                         2.419e+04
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                                          coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------
In

In [15]:
# Step 4: Generating predictions
# We extract the predicted values vector to transition from explanation to prediction
y_pred = results.predict(df)

In [16]:
# Step 5: Calculate RMSE between the actuals and the predictions
model_rmse = rmse(df['Home_Value'], y_pred)
print(f"\nThe Predictive RMSE is: ${model_rmse:,.2f}")


The Predictive RMSE is: $42,316.69


In [17]:
"""
=============================================================================
  RESIDUAL FORENSICS DASHBOARD
  Hedonic Pricing OLS Model — Interactive Outlier & Heteroscedasticity Probe
  Built with: statsmodels · plotly.express · pandas · numpy
=============================================================================
"""

import numpy as np
import pandas as pd
import statsmodels.api as sm
import plotly.express as px
import plotly.graph_objects as go

# =============================================================================
# SECTION 1: SYNTHETIC DATA & OLS MODEL
# (Replace this block with your own model results object — see note below)
# =============================================================================

np.random.seed(42)
n = 300

# Simulate hedonic pricing features (e.g., square footage, bedrooms, age)
sqft      = np.random.normal(1800, 400, n)
bedrooms  = np.random.randint(2, 6, n)
age       = np.random.randint(1, 50, n)

# Inject mild heteroscedasticity: variance grows with sqft
noise = np.random.normal(0, 0.05 * sqft, n)

log_price = (
    10.5
    + 0.0004 * sqft
    + 0.08   * bedrooms
    - 0.003  * age
    + noise
)

df = pd.DataFrame({
    "log_price": log_price,
    "sqft":      sqft,
    "bedrooms":  bedrooms,
    "age":       age,
})

# Build the design matrix (statsmodels requires explicit constant)
X = sm.add_constant(df[["sqft", "bedrooms", "age"]])
y = df["log_price"]

# Fit OLS — `results` is the statsmodels RegressionResultsWrapper
results = sm.OLS(y, X).fit()

print(results.summary())

# =============================================================================
# SECTION 2: EXTRACT RESIDUALS FROM THE STATSMODELS RESULTS OBJECT
# =============================================================================

# `results.fittedvalues` → pandas Series of ŷ  (in-sample predicted values)
# This is equivalent to X @ results.params but already computed and cached.
fitted = results.fittedvalues          # ŷ

# `results.resid` → pandas Series of raw OLS residuals: e = y − ŷ
# statsmodels stores this as a named attribute on the results object;
# no manual subtraction needed. These are the *unstandardized* residuals.
residuals = results.resid              # e = y − ŷ

# =============================================================================
# SECTION 3: OUTLIER CLASSIFICATION  (|z| > 2σ rule)
# =============================================================================

# Compute the standard deviation of residuals across all observations
resid_std = residuals.std()            # scalar σ

# Z-score each residual relative to the residual distribution
# A z-score > 2 means the observation sits more than 2σ from the zero line
z_scores = residuals / resid_std       # element-wise standardisation

# Boolean mask: True where |z| exceeds the 2σ threshold
is_outlier = z_scores.abs() > 2.0

# Human-readable label for the color encoding in Plotly
outlier_label = np.where(is_outlier, "Outlier  |z| > 2σ", "Normal")

# =============================================================================
# SECTION 4: ASSEMBLE PLOT DATAFRAME
# =============================================================================

plot_df = pd.DataFrame({
    "Fitted Values (ŷ)":   fitted.values,
    "Residuals (e)":        residuals.values,
    "Z-Score":              z_scores.values,
    "Outlier":              outlier_label,
    # Hover extras — replace with your own feature names if desired
    "sqft":                 df["sqft"].values,
    "bedrooms":             df["bedrooms"].values,
    "age":                  df["age"].values,
})

# =============================================================================
# SECTION 5: BUILD THE INTERACTIVE PLOTLY SCATTER
# =============================================================================

# --- Color map: normal points use a steel-blue; outliers get stark crimson ---
color_discrete_map = {
    "Normal":          "#4A90D9",   # calm steel-blue for inliers
    "Outlier  |z| > 2σ": "#DC143C", # stark crimson for flagged observations
}

fig = px.scatter(
    plot_df,
    x="Fitted Values (ŷ)",
    y="Residuals (e)",
    color="Outlier",                         # encode outlier status via hue
    color_discrete_map=color_discrete_map,
    opacity=0.75,
    size_max=10,
    # Rich hover card — shows z-score and raw features on mouse-over
    hover_data={
        "Z-Score":              ":.3f",
        "sqft":                 ":.0f",
        "bedrooms":             True,
        "age":                  True,
        "Fitted Values (ŷ)":   ":.4f",
        "Residuals (e)":        ":.4f",
    },
    title="<b>Residual Forensics Dashboard</b>  —  OLS Hedonic Pricing Model",
    labels={
        "Fitted Values (ŷ)": "Fitted Values  ŷ",
        "Residuals (e)":      "Residuals  e = y − ŷ",
    },
    template="plotly_dark",
)

# --- Add horizontal zero-reference line (ideal: residuals centred on 0) ---
# `add_hline` is the current, non-deprecated approach (replaces layout.shapes)
fig.add_hline(
    y=0,
    line_dash="dash",
    line_color="rgba(255,255,255,0.55)",
    line_width=1.5,
    annotation_text="  Zero Line (e = 0)",
    annotation_position="bottom right",
    annotation_font_color="rgba(255,255,255,0.55)",
)

# --- Add ±2σ band as horizontal lines to visualise the outlier threshold ---
for sign, label in [(+1, "+2σ"), (-1, "−2σ")]:
    fig.add_hline(
        y=sign * 2 * resid_std,
        line_dash="dot",
        line_color="rgba(220,20,60,0.45)",   # faint crimson band
        line_width=1.2,
        annotation_text=f"  {label} threshold",
        annotation_position="top right" if sign > 0 else "bottom right",
        annotation_font_color="rgba(220,20,60,0.75)",
    )

# --- Style refinements ----------------------------------------------------- #
fig.update_traces(marker=dict(size=7, line=dict(width=0.4, color="white")))

fig.update_layout(
    font_family="'IBM Plex Mono', monospace",
    title_font_size=17,
    legend_title_text="Observation Class",
    legend=dict(
        bgcolor="rgba(255,255,255,0.06)",
        bordercolor="rgba(255,255,255,0.15)",
        borderwidth=1,
    ),
    xaxis=dict(showgrid=True, gridcolor="rgba(255,255,255,0.07)", zeroline=False),
    yaxis=dict(showgrid=True, gridcolor="rgba(255,255,255,0.07)", zeroline=False),
    margin=dict(l=60, r=40, t=80, b=60),
    height=620,
)

# =============================================================================
# SECTION 6: SUMMARY ANNOTATION BOX (RMSE + OUTLIER COUNT)
# =============================================================================

# RMSE pulled directly from the results object
rmse         = np.sqrt(results.mse_resid)   # mse_resid = σ² of the regression
n_outliers   = int(is_outlier.sum())
pct_outliers = 100 * n_outliers / n

# Inject a text annotation into the chart (top-left corner)
fig.add_annotation(
    x=0.01, y=0.98,
    xref="paper", yref="paper",
    text=(
        f"<b>Model Diagnostics</b><br>"
        f"RMSE : {rmse:.5f}<br>"
        f"R²   : {results.rsquared:.4f}<br>"
        f"N    : {n}<br>"
        f"Outliers (|z|>2σ) : {n_outliers} ({pct_outliers:.1f}%)"
    ),
    align="left",
    showarrow=False,
    bgcolor="rgba(0,0,0,0.45)",
    bordercolor="rgba(255,255,255,0.2)",
    borderwidth=1,
    font=dict(size=11, color="white", family="'IBM Plex Mono', monospace"),
)

# =============================================================================
# SECTION 7: RENDER
# =============================================================================

fig.show()   # Opens in default browser; swap for fig.write_html("dashboard.html")
             # or fig.write_image("dashboard.png") for static export.


# =============================================================================
# USAGE NOTE
# =============================================================================
# If you already have a fitted statsmodels results object from your own lab,
# simply replace sections 1–2 with:
#
#   results  = <your existing RegressionResultsWrapper>
#   fitted   = results.fittedvalues
#   residuals = results.resid
#
# Everything from Section 3 onward is model-agnostic.
# =============================================================================

                            OLS Regression Results                            
Dep. Variable:              log_price   R-squared:                       0.010
Model:                            OLS   Adj. R-squared:                 -0.000
Method:                 Least Squares   F-statistic:                    0.9881
Date:                Mon, 16 Mar 2026   Prob (F-statistic):              0.399
Time:                        20:03:04   Log-Likelihood:                -1804.9
No. Observations:                 300   AIC:                             3618.
Df Residuals:                     296   BIC:                             3633.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         62.7698     32.252      1.946      0.0